# Logging scikit-learn models well — safe serialization & `autolog()`

*Part of the `ml/` track. Prerequisite: `basics/b_tracking_quickstart`.*

In `basics/b_tracking_quickstart` you logged a model, loaded it back as a `pyfunc`, and predicted with it — the flow that's shared by every framework. This notebook covers two concerns specific to **scikit-learn** model logging:

- **Safe serialization** — why MLflow warns about pickle, and the one-line `skops` fix.
- **`mlflow.sklearn.autolog()`** — the one-line alternative to logging params, metrics, and the model by hand.

### Prerequisites

A tracking server on port **5001** (the `ml/` track's port). Start it from `src/` in a separate terminal and leave it running:

```bash
mlflow server --host 127.0.0.1 --port 5001
```

In [ ]:
import mlflow
from mlflow.models import infer_signature
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("ml-model-logging")

In [ ]:
# Train a small stock-sklearn model to log (the same iris example as basics/b_).
X, y = datasets.load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {"solver": "lbfgs", "max_iter": 1000, "random_state": 42}
lr = LogisticRegression(**params).fit(X_train, y_train)
accuracy = accuracy_score(y_test, lr.predict(X_test))
print(f"accuracy: {accuracy:.3f}")

## Serializing with `skops` instead of pickle

**Stale in upstream tutorial:**  
When you run `mlflow.sklearn.log_model(...)` without specifying `serialization_format`, MLflow emits this warning:

```
WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle
format requires exercising caution because these formats rely on Python's object
serialization mechanism, which can execute arbitrary code during deserialization.
The recommended safe alternative is the 'skops' format.
```

This section explains **what the warning means**, **why it matters**, and **the one-line fix** shown below.

### Why pickle/cloudpickle are unsafe

`pickle` (and its superset `cloudpickle`, which adds support for lambdas, locally-defined classes, and other "tricky" Python objects) does not serialize *data*.  
It serializes a **byte-stream program** that, when read back, reconstructs the object by executing instructions on a virtual machine inside the unpickler.

Those instructions include opcodes like:

- `GLOBAL` — look up any module/attribute by name  
- `REDUCE` — call any callable with any arguments  
- `BUILD` — set arbitrary state on the result

In practice this means: **anyone who can hand you a `.pkl` file can run arbitrary Python on your machine the moment you call `mlflow.pyfunc.load_model(...)`**.  
No exploit, no parser bug — that *is* the documented behavior of `pickle.load`.

The Python docs are explicit:

> **Warning:** The `pickle` module **is not secure**. Only unpickle data you trust.

For a personal notebook on your own laptop this is fine: you trained the model, you saved it, you loaded it.  
But the moment a model moves between people or systems — a teammate pulls it from the registry, a CI job loads it from S3, you publish it to a public artifact store — pickle-based loading becomes a real attack surface.

### What `skops` does differently

`skops` is a serialization library from the scikit-learn org, built specifically to replace pickle for ML models.  
Its persistence format is fundamentally different:

| Property | `pickle` / `cloudpickle` | `skops` |
|---|---|---|
| On-disk shape | Byte-stream program | Zip archive: numpy arrays + JSON schema |
| Loading mechanism | Executes serialized opcodes | Reconstructs known types from a schema |
| Unknown types | Silently instantiated (RCE) | Refused unless explicitly trusted |
| File extension | `.pkl` | `.skops` |
| Diff-friendly | No (binary opcodes) | Mostly yes (JSON-ish schema) |

**The key idea:** skops loads from a **declarative description** of the object tree, not an imperative program.  
When the loader sees `sklearn.linear_model.LogisticRegression`, it checks that type against an allow-list of known-safe types and instantiates it.  
Anything **not** on the allow-list raises `UntrustedTypesFoundException` instead of being executed.

### The fix

Two changes were needed:

**1. Add `skops` to the project dependencies (uv):**

```bash
uv add skops
```

This added `skops>=0.14.0` to `[project].dependencies` in `pyproject.toml` and recorded the resolved version in `uv.lock`.

**2. Pass `serialization_format="skops"` to `log_model`** (the run cell below).  
MLflow's `mlflow.sklearn.log_model` accepts three values for this argument: `"pickle"`, `"cloudpickle"` (the default), and `"skops"`.

That's the whole fix.  
For built-in scikit-learn estimators like `LogisticRegression`, the skops default allow-list already includes every required type, so `mlflow.pyfunc.load_model(...)` later in this notebook still works with no other changes.

### Caveat: custom classes

If your model contains any **non-scikit-learn class** — a custom `BaseEstimator` subclass, a custom `Transformer`, a pipeline step you defined yourself — `skops` will refuse to load it by default, because that class isn't in its allow-list.

The escape hatch is the `skops_trusted_types` argument:

```python
mlflow.sklearn.log_model(
    sk_model=pipe,
    name="pipe_model",
    serialization_format="skops",
    skops_trusted_types=["__main__.MyCustomTransformer"],
)
```

This is *opt-in trust*, not "turn off all checks".  
You're stating, per fully-qualified type name, that you accept responsibility for that class being safe to instantiate.  
The iris model in this notebook is pure stock sklearn, so no `skops_trusted_types` list is needed.

### When you might *not* use `skops`

A few situations still call for cloudpickle (and accepting the warning consciously):

- **Models that pickle closures or lambdas.**  
  skops's format is structural; it cannot serialize a function body.  
  If your estimator captures a lambda, you'll get a `NotImplementedError` from skops and need cloudpickle.  
- **Third-party estimators with deeply custom internals** — some libraries (e.g. certain gradient-boosting wrappers) store state that skops can't introspect.  
- **Quick local experimentation where the model file never leaves your machine** — the security risk is zero, and pickle is one less dependency.

For everything else — and especially for any model that will live in the MLflow registry where other people or services will load it — `skops` is the correct default in MLflow 3+.

### References

- [`mlflow.sklearn` API — `serialization_format` parameter](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html)  
- [Pickle-Free Model format — MLflow docs](https://mlflow.org/docs/latest/ml/tracking/pickle-free-models/)  
- [`skops` project — secure scikit-learn serialization](https://skops.readthedocs.io/en/stable/persistence.html)

In [ ]:
# Log the trained model with the safe skops format (see the explanation above).
with mlflow.start_run(run_name="skops-serialized"):
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy)
    signature = infer_signature(X_train, lr.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train,
        serialization_format="skops",
    )
print(model_info.model_uri)

## The one-line alternative: `mlflow.autolog()`

Everything in the skops run cell above was **explicit**: you called `log_params`, `log_metric`, and `log_model` by hand, so you could see exactly what gets recorded. `mlflow.autolog()` does the same logging **automatically** — it *patches* the training library so that the moment you call `.fit()`, MLflow records the hyperparameters, a standard set of metrics, the fitted model, and an inferred signature, with no logging calls in your code at all.

**The problem it solves:** manual logging is precise but easy to get wrong — forget one `log_param`, mistype a metric name, or skip the signature, and your run is silently incomplete. Across many experiments and teammates, those omissions are how a run becomes irreproducible. Autolog trades fine control for a *can't-forget* guarantee: one line, and the standard record is always captured.

In [ ]:
# mlflow.autolog() works by PATCHING the training library: once enabled, calling `.fit()`
# automatically logs params, metrics, the model, and a signature — no explicit
# log_params / log_metric / log_model calls.
#
# Two deliberate choices keep this demo tidy:
#   * a separate experiment, so these auto-logged runs don't mix with the manual ones above;
#   * NO registered_model_name, so the "tracking-quickstart" registered model isn't bumped
#     to a new version just by running this section.
mlflow.set_experiment("MLflow Quickstart (autolog)")

# Flavor-specific switch. The bare `mlflow.autolog()` would enable this for EVERY supported
# library at once (xgboost, lightgbm, pytorch, …); `mlflow.sklearn.autolog()` is the same
# mechanism scoped to scikit-learn. Flavor-specific settings take precedence over the bare call.
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="iris-autolog-demo") as run:
    # Just fit. There is not a single explicit MLflow call inside this block.
    LogisticRegression(**params).fit(X_train, y_train)

autolog_run_id = run.info.run_id
print("autolog run:", autolog_run_id)

# Turn patching back off so later cells in this notebook log only what they ask for.
mlflow.sklearn.autolog(disable=True)

### What landed automatically — and three gotchas

Open the `iris-autolog-demo` run in the UI and compare it to the manual run from earlier. From that single `.fit()`, autolog captured:

| | Manual run (above) | Autolog run |
|---|---|---|
| Parameters | the 3 you put in `params` | **every** `get_params()` value, including sklearn defaults you never set |
| Metrics | the one `accuracy` you computed | a standard set of `training_*` metrics — accuracy, F1, precision, recall, log-loss, ROC-AUC, and the estimator's own `score` — on the training data |
| Model + signature | logged explicitly | logged automatically, signature inferred |
| Dataset | not logged | training dataset logged (`log_datasets=True` by default) |

Convenience, but read the fine print — three defaults bite beginners:

1. **No input example unless you ask.** `log_input_examples=False` by default, so unlike the manual run there's no `input_example` saved. That example is what makes a *served* endpoint self-describing (you'll see why in `g_model_serving`); opt in with `mlflow.sklearn.autolog(log_input_examples=True)`.
2. **It reverts to `cloudpickle` — the format the skops section just warned about.** Autolog's default `serialization_format="cloudpickle"` means the pickle security warning is back. The flavor call can still take `mlflow.sklearn.autolog(serialization_format="skops")`; the bare `mlflow.autolog()` cannot. Convenience costs the safe-by-default choice you made by hand.
3. **It logs *more* than you might want.** This very run auto-logged an `estimator.html` summary and a `training_confusion_matrix.png` — artifacts you never asked for (check the run's *Artifacts* tab). Great for an audit trail, noisy if you only cared about two numbers.

### Manual vs autolog — when to reach for which

- **Manual** when the log is a *contract*: production training scripts where you want exactly these metric names, the skops-safe format, and nothing surprising. You saw the whole surface in this notebook.
- **Autolog** for *breadth and speed*: rapid experimentation where the can't-forget guarantee matters more than precise control.
- **They coexist.** Inside an autolog run you can still add manual `mlflow.log_metric(...)` calls — autolog fills in the standard record, you add the bespoke bits on top.

**Next:** `b_hyperparameter_tuning` shows autolog's real payoff for search — point it at a `GridSearchCV` and (via `max_tuning_runs`) it logs the parent best-run plus child runs for the top parameter sets automatically, instead of the manual child-run bookkeeping you'd otherwise write.

## Next steps

- **`b_hyperparameter_tuning.ipynb`** — scale from logging one model to a search over many, with parent/child runs and an Optuna sweep. Its autolog aside builds directly on the `mlflow.sklearn.autolog()` you just saw.